In [0]:
doctors_df = spark.read.csv(
    '/Volumes/learn_databricks/default/datasets/doctors.csv',
    header=True,
    inferSchema=True
)

visits_df = spark.read.csv(
    '/Volumes/learn_databricks/default/datasets/visits.csv',
    header=True,
    inferSchema=True
)

display(doctors_df)
display(visits_df)

doctor_id,doctor_name,specialization,city,experience_years,consultation_fee
D101,Dr. Ramesh,Cardiology,Hyderabad,12,1500
D102,Dr. Priya,Neurology,Bangalore,10,2000
D103,Dr. Anita,Dermatology,Chennai,8,1000
D104,Dr. Suresh,Orthopedics,Mumbai,15,2500
D105,Dr. Meera,Pediatrics,Delhi,6,1200
D106,Dr. Kiran,Cardiology,Hyderabad,20,3000
D107,Dr. Farhan,Neurology,Pune,5,1800
D108,Dr. Nisha,Dermatology,Kochi,9,1100


visit_id,patient_name,doctor_id,visit_date,diagnosis,bill_amount,payment_status
V1001,Rahul Sharma,D101,2026-01-10,Heart Checkup,5000,Paid
V1002,Priya Reddy,D102,2026-01-10,Migraine,3500,Paid
V1003,Amit Kumar,D103,2026-01-11,Skin Allergy,2000,Pending
V1004,Sneha Patel,D104,2026-01-11,Fracture,12000,Paid
V1005,Farhan Ali,D105,2026-01-12,Fever,1500,Paid
V1006,Neha Singh,D106,2026-01-12,Heart Checkup,7000,Paid
V1007,Arjun Verma,D120,2026-01-13,Migraine,4000,Paid
V1008,Meera Nair,D103,2026-01-13,Skin Allergy,null,Pending
V1009,Kiran Rao,D104,2026-01-14,Back Pain,6000,Paid
V1010,Nisha Reddy,D101,2026-01-14,Heart Checkup,5500,Paid


In [0]:
from pyspark.sql.functions import *

doctors_df.printSchema()
visits_df.printSchema()

print('doctor rec count : ', doctors_df.count())
print('visit rec count : ', visits_df.count())

doctors_df.filter(col('city')=='Hyderabad').show()
doctors_df.filter(col('specialization')=='Cardiology').show()
doctors_df.filter(col('experience_years')>10).show()
visits_df.filter(col('bill_amount')>5000).show()
visits_df.filter(col('payment_status')=='Pending').show()
visits_df.filter(col('payment_status')=='Paid').show()
doctors_df.groupBy('specialization').agg(avg('consultation_fee')).show()
doctors_df.groupBy('specialization').agg(max('consultation_fee')).show()
doctors_df.groupBy('city').count().show()
doctors_df.groupBy('specialization').count().show()
visits_df.filter(col('payment_status')=='Paid').agg(sum('bill_amount')).show()
visits_df.agg(avg('bill_amount')).show()
visits_df.agg(max('bill_amount')).show()
visits_df.agg(min('bill_amount')).show()
doctors_df.orderBy(col('consultation_fee').desc()).show()
visits_df.orderBy(col('bill_amount').desc()).show()
visits_df.filter(col('bill_amount').isNull()).show()
visits_df = visits_df.na.fill({
    'bill_amount': 0
})
full_visits_df = (visits_df.withColumn(
    'tax',
    col('bill_amount') * 0.05
).withColumn(
    'final_bill',
    col("bill_amount") + col('tax')
))

full_visits_df.show()

root
 |-- doctor_id: string (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- specialization: string (nullable = true)
 |-- city: string (nullable = true)
 |-- experience_years: integer (nullable = true)
 |-- consultation_fee: integer (nullable = true)

root
 |-- visit_id: string (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- visit_date: date (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- bill_amount: integer (nullable = true)
 |-- payment_status: string (nullable = true)

doctor rec count :  8
visit rec count :  10
+---------+-----------+--------------+---------+----------------+----------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|
+---------+-----------+--------------+---------+----------------+----------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|    

In [0]:
doctors_df.join(
    visits_df,
    "doctor_id",
    'inner'
).show()
doctors_df.join(
    visits_df,
    'doctor_id',
    'left'
).show()
doctors_df.join(
    visits_df,
    'doctor_id',
    'right'
).show()
doctors_df.join(
    visits_df,
    'doctor_id',
    'full'
).show()
visits_df.join(
    doctors_df,
    'doctor_id',
    'left_anti'
).show()
doctors_df.join(
    visits_df,
    'doctor_id',
    'left_anti'
).show()

doctors_df.join(
    visits_df,
    'doctor_id',
    'left'
).groupby('doctor_id').agg(count('visit_id').alias('visit count')).show()

doctors_df.join(
    visits_df,
    'doctor_id',
    'left'
).groupby('doctor_id').agg(sum('bill_amount').alias('revenue')).show()

doctors_df.join(
    visits_df,
    'doctor_id',
    'left'
).groupBy('doctor_id').agg(sum('bill_amount').alias('revenue')
).orderBy(col('revenue').desc()).show(1)

doctors_df.join(
    visits_df,
    'doctor_id',
    'left'
).groupBy('specialization').agg(sum('bill_amount').alias('revenue')
).orderBy(col('revenue').desc()).show(1)

doctors_df.join(
    visits_df,
    'doctor_id',
    'left'
).groupby('city').agg(sum('bill_amount').alias('revenue')).show()

doctors_df.join(
    visits_df,
    'doctor_id',
    'left'
).groupby('doctor_id').agg(count('visit_id').alias('visit count')).show()

doctors_df.join(
    visits_df,
    'doctor_id',
    'left'
).groupBy('doctor_id').agg(sum('bill_amount').alias('revenue')
).orderBy(col('revenue').desc()).show(3)

performance_report_df = doctors_df.join(
    visits_df,
    "doctor_id",
    "left"
).groupBy(
    "doctor_id",
    "doctor_name",
    "specialization",
    "city"
).agg(
    count("visit_id").alias("visit_count"),
    sum("bill_amount").alias("total_revenue"),
    avg("bill_amount").alias("avg_revenue")
).orderBy(
    col("total_revenue").desc()
)

performance_report_df.show(truncate=False)

+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_status|
+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|   V1001|Rahul Sharma|2026-01-10|Heart Checkup|       5000|          Paid|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|            2000|   V1002| Priya Reddy|2026-01-10|     Migraine|       3500|          Paid|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|            1000|   V1003|  Amit Kumar|2026-01-11| Skin Allergy|       2000|       Pending|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|          

In [0]:
hospital_config_df = spark.read.option(
    'multiline',
    'true'
).json('/Volumes/learn_databricks/default/datasets/hospital_config.json')

hospital_config_df.printSchema()

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- hospital_id: string (nullable = true)
 |-- hospital_name: string (nullable = true)
 |-- services: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [0]:
flat_hospital_config_df = hospital_config_df.select(
    'city','hospital_id','hospital_name','services',
    'contact.email', 'contact.phone'
)

flat_hospital_config_df.filter(col('phone').isNull()).show()
flat_hospital_config_df.filter(col('email').isNull()).show()
flat_hospital_config_df = flat_hospital_config_df.na.fill({
    'phone': 0000000000,
    'email': 'example@email.com'
})

flat_hospital_config_df.select('hospital_name', 'city').show()
flat_hospital_config_df.select('hospital_name', 'phone').show()
flat_hospital_config_df.select(
    'hospital_name',
    'city',
    explode('services').alias('service')
).show()



+---------+-----------+----------------+--------------------+----------------+-----+
|     city|hospital_id|   hospital_name|            services|           email|phone|
+---------+-----------+----------------+--------------------+----------------+-----+
|Hyderabad|       H102|Yashoda Hospital|[Cardiology, Orth...|yashoda@mail.com| NULL|
+---------+-----------+----------------+--------------------+----------------+-----+

+---------+-----------+-------------+--------------------+-----+----------+
|     city|hospital_id|hospital_name|            services|email|     phone|
+---------+-----------+-------------+--------------------+-----+----------+
|Bangalore|       H103|Care Hospital|[Neurology, Pedia...| NULL|9876500013|
+---------+-----------+-------------+--------------------+-----+----------+

+----------------+---------+
|   hospital_name|     city|
+----------------+---------+
| Apollo Hospital|Hyderabad|
|Yashoda Hospital|Hyderabad|
|   Care Hospital|Bangalore|
+----------------+-

In [0]:
hospital_services_df = hospital_config_df.select(
    'hospital_id',
    'hospital_name',
    'city',
    explode('services').alias('service')
)

hospital_services_df.show()

hospital_config_df.groupBy("city").count().show()
hospital_services_df.groupBy("service").count().show()
hospital_services_df.filter(col("service") == "Cardiology").show()
hospital_services_df.filter(col("service") == "Neurology").show()
hospital_services_df.filter(col("service") == "Orthopedics").show()
hospital_services_df.filter(col("service") == "Pediatrics").show()

+-----------+----------------+---------+-----------+
|hospital_id|   hospital_name|     city|    service|
+-----------+----------------+---------+-----------+
|       H101| Apollo Hospital|Hyderabad| Cardiology|
|       H101| Apollo Hospital|Hyderabad|  Neurology|
|       H101| Apollo Hospital|Hyderabad|Dermatology|
|       H102|Yashoda Hospital|Hyderabad| Cardiology|
|       H102|Yashoda Hospital|Hyderabad|Orthopedics|
|       H103|   Care Hospital|Bangalore|  Neurology|
|       H103|   Care Hospital|Bangalore| Pediatrics|
+-----------+----------------+---------+-----------+

+---------+-----+
|     city|count|
+---------+-----+
|Hyderabad|    2|
|Bangalore|    1|
+---------+-----+

+-----------+-----+
|    service|count|
+-----------+-----+
|Orthopedics|    1|
| Cardiology|    2|
| Pediatrics|    1|
|Dermatology|    1|
|  Neurology|    2|
+-----------+-----+

+-----------+----------------+---------+----------+
|hospital_id|   hospital_name|     city|   service|
+-----------+---------

In [0]:
hospital_services_df.write \
    .mode("overwrite") \
    .parquet("/Volumes/learn_databricks/default/generated_data/hospital_services_parquet")

hospital_services_df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/Volumes/learn_databricks/default/generated_data/hospital_services_csv")

In [0]:
from pyspark.sql.window import Window

doctor_revenue_df = doctors_df.join(
    visits_df,
    "doctor_id",
    "left"
).groupBy(
    "doctor_id",
    "doctor_name",
    "specialization",
    "city"
).agg(
    sum("bill_amount").alias("revenue")
)

In [0]:
revenue_window = Window.orderBy(col("revenue").desc())

doctor_revenue_df.withColumn(
    "rank",
    rank().over(revenue_window)
).show()

doctor_revenue_df.withColumn(
    "dense_rank",
    dense_rank().over(revenue_window)
).show()

doctor_revenue_df.withColumn(
    "row_number",
    row_number().over(revenue_window)
).show()

doctor_revenue_df.orderBy(col("revenue").desc()).show(1)

doctor_revenue_df.orderBy(col("revenue").desc()).show(3)

spec_window = Window.partitionBy("specialization").orderBy(
    col("revenue").desc()
)

doctor_revenue_df.withColumn(
    "rank",
    rank().over(spec_window)
).filter(col("rank") == 1).show()

doctor_revenue_df.withColumn(
    "rank",
    rank().over(spec_window)
).filter(col("rank") <= 2).show()

running_window = Window.orderBy(
    col("revenue").desc()
).rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow
)

doctor_revenue_df.withColumn(
    "running_revenue",
    sum("revenue").over(running_window)
).show()

doctor_revenue_df.withColumn(
    "previous_revenue",
    lag("revenue", 1).over(revenue_window)
).show()

doctor_revenue_df.withColumn(
    "next_revenue",
    lead("revenue", 1).over(revenue_window)
).show()

doctor_revenue_df.withColumn(
    "previous_revenue",
    lag("revenue", 1).over(revenue_window)
).withColumn(
    "difference_from_previous",
    col("revenue") - col("previous_revenue")
).show()

doctor_revenue_df.withColumn(
    "next_revenue",
    lead("revenue", 1).over(revenue_window)
).withColumn(
    "difference_from_next",
    col("revenue") - col("next_revenue")
).show()

city_window_desc = Window.partitionBy("city").orderBy(col("revenue").desc())

doctor_revenue_df.withColumn(
    "rank",
    rank().over(city_window_desc)
).filter(col("rank") == 1).show()

city_window_asc = Window.partitionBy("city").orderBy(col("revenue").asc())

doctor_revenue_df.withColumn(
    "rank",
    rank().over(city_window_asc)
).filter(col("rank") == 1).show()

leaderboard_df = doctor_revenue_df.withColumn(
    "rank",
    dense_rank().over(revenue_window)
).orderBy(col("rank"))

leaderboard_df.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|   1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|   2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|   3|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|   4|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|   5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|   6|
|     D108|  Dr. Nisha|   Dermatology|    Kochi|   NULL|   7|
|     D107| Dr. Farhan|     Neurology|     Pune|   NULL|   7|
+---------+-----------+--------------+---------+-------+----+

+---------+-----------+--------------+---------+-------+----------+
|doctor_id|doctor_name|specialization|     city|revenue|dense_rank|
+---------+-----------+--------------+---------+-------+----------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  1

In [0]:
doctors_df.createOrReplaceTempView("doctors")
visits_df.createOrReplaceTempView("visits")
hospital_config_df.createOrReplaceTempView("hospitals")

spark.sql("""
    SELECT *
    FROM doctors
""").show()

spark.sql("""
    SELECT specialization,
        COUNT(*) AS doctor_count
    FROM doctors
    GROUP BY specialization
""").show()

spark.sql("""
    SELECT city,
        COUNT(*) AS doctor_count
    FROM doctors
    GROUP BY city
""").show()

spark.sql("""
    SELECT d.doctor_id,
        d.doctor_name,
        SUM(v.bill_amount) AS revenue
    FROM doctors d
    LEFT JOIN visits v
    ON d.doctor_id = v.doctor_id
    GROUP BY d.doctor_id,
            d.doctor_name
    ORDER BY revenue DESC
""").show()

spark.sql("""
    SELECT d.specialization,
        SUM(v.bill_amount) AS revenue
    FROM doctors d
    LEFT JOIN visits v
    ON d.doctor_id = v.doctor_id
    GROUP BY d.specialization
    ORDER BY revenue DESC
""").show()

spark.sql("""
    SELECT d.doctor_id,
        d.doctor_name,
        SUM(v.bill_amount) AS revenue
    FROM doctors d
    LEFT JOIN visits v
    ON d.doctor_id = v.doctor_id
    GROUP BY d.doctor_id,
            d.doctor_name
    ORDER BY revenue DESC
    LIMIT 5
""").show()

spark.sql("""
    SELECT *
    FROM visits
    WHERE payment_status = 'Pending'
""").show()

spark.sql("""
    SELECT *
    FROM hospitals
    WHERE array_contains(services, 'Cardiology')
""").show(truncate=False)

spark.sql("""
    SELECT *
    FROM hospitals
    WHERE array_contains(services, 'Neurology')
""").show(truncate=False)

spark.sql("""
    SELECT *
    FROM hospitals
    WHERE contact.phone IS NULL
    OR contact.email IS NULL
""").show(truncate=False)

spark.sql("""
    SELECT AVG(consultation_fee) AS avg_consultation_fee
    FROM doctors
""").show()

spark.sql("""
    SELECT
        d.doctor_id,
        d.doctor_name,
        d.specialization,
        d.city,
        COUNT(v.visit_id) AS visit_count,
        SUM(v.bill_amount) AS total_revenue,
        AVG(v.bill_amount) AS avg_revenue
    FROM doctors d
    LEFT JOIN visits v
    ON d.doctor_id = v.doctor_id
    GROUP BY
        d.doctor_id,
        d.doctor_name,
        d.specialization,
        d.city
    ORDER BY total_revenue DESC
""").show(truncate=False)

+---------+-----------+--------------+---------+----------------+----------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|
+---------+-----------+--------------+---------+----------------+----------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|            2000|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|            1000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|            2500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|               6|            1200|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|            3000|
|     D107| Dr. Farhan|     Neurology|     Pune|               5|            1800|
|     D108|  Dr. Nisha|   Dermatology|    Kochi|               9|            1100|
+---------+-----------+--------------+---------+----------------+----------------+

+--

In [0]:
doctors_df = spark.read.option(
    "header", True
).option(
    "inferSchema", True
).csv(
    "/Volumes/learn_databricks/default/datasets/doctors.csv"
)

visits_df = spark.read.option(
    "header", True
).option(
    "inferSchema", True
).csv(
    "/Volumes/learn_databricks/default/datasets/visits.csv"
)

hospital_df = spark.read.option(
    "multiline", True
).json(
    "/Volumes/learn_databricks/default/datasets/hospital_config.json"
)

visits_df = visits_df.na.fill({
    "bill_amount": 0
})

hospital_df = hospital_df.na.fill({
    "city": "Unknown"
})

hospital_services_df = hospital_df.select(
    "hospital_id",
    "hospital_name",
    "city",
    explode("services").alias("service")
)

doctor_visit_df = doctors_df.join(
    visits_df,
    "doctor_id",
    "left"
)

doctor_visit_df = doctor_visit_df.withColumn(
    "tax",
    col("bill_amount") * 0.05
).withColumn(
    "final_bill",
    col("bill_amount") + col("tax")
)

doctor_revenue_df = doctor_visit_df.groupBy(
    "doctor_id",
    "doctor_name",
    "specialization",
    "city"
).agg(
    sum("bill_amount").alias("revenue")
)

ranking_window = Window.orderBy(
    col("revenue").desc()
)

doctor_rank_df = doctor_revenue_df.withColumn(
    "doctor_rank",
    dense_rank().over(ranking_window)
)

specialization_summary_df = doctor_visit_df.groupBy(
    "specialization"
).agg(
    countDistinct("doctor_id").alias("doctor_count"),
    count("visit_id").alias("visit_count"),
    sum("bill_amount").alias("total_revenue"),
    avg("bill_amount").alias("avg_revenue")
)

doctor_visit_df.write.mode(
    "overwrite"
).parquet(
    "/Volumes/learn_databricks/default/generated_data/silver/doctor_visit"
)

hospital_services_df.write.mode(
    "overwrite"
).parquet(
    "/Volumes/learn_databricks/default/generated_data/silver/hospital_services"
)

doctor_rank_df.write.mode(
    "overwrite"
).parquet(
    "/Volumes/learn_databricks/default/generated_data/gold/doctor_ranking"
)

specialization_summary_df.write.mode(
    "overwrite"
).parquet(
    "/Volumes/learn_databricks/default/generated_data/gold/specialization_summary"
)

dashboard_df = doctor_visit_df.groupBy(
    "city",
    "specialization"
).agg(
    countDistinct("doctor_id").alias("doctor_count"),
    count("visit_id").alias("visit_count"),
    sum("bill_amount").alias("total_revenue"),
    avg("bill_amount").alias("avg_bill_amount"),
    max("bill_amount").alias("highest_bill_amount")
)

dashboard_df.write.mode(
    "overwrite"
).parquet(
    "/Volumes/learn_databricks/default/generated_data/gold/hospital_dashboard"
)

dashboard_df.show(truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+--------------+------------+-----------+-------------+-----------------+-------------------+
|city     |specialization|doctor_count|visit_count|total_revenue|avg_bill_amount  |highest_bill_amount|
+---------+--------------+------------+-----------+-------------+-----------------+-------------------+
|Pune     |Neurology     |1           |0          |NULL         |NULL             |NULL               |
|Hyderabad|Cardiology    |2           |3          |17500        |5833.333333333333|7000               |
|Kochi    |Dermatology   |1           |0          |NULL         |NULL             |NULL               |
|Chennai  |Dermatology   |1           |2          |2000         |1000.0           |2000               |
|Mumbai   |Orthopedics   |1           |2          |18000        |9000.0           |12000              |
|Bangalore|Neurology     |1           |1          |3500         |3500.0           |3500               |
|Delhi    |Pediatrics    |1           |1          |1500         

In [0]:
doctors_df.printSchema()
visits_df.printSchema()

root
 |-- doctor_id: string (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- specialization: string (nullable = true)
 |-- city: string (nullable = true)
 |-- experience_years: integer (nullable = true)
 |-- consultation_fee: integer (nullable = true)

root
 |-- visit_id: string (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- visit_date: date (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- bill_amount: integer (nullable = false)
 |-- payment_status: string (nullable = true)

